<a href="https://colab.research.google.com/github/GARVIT-bit/BIS/blob/main/Geneticsoptimisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# ==========================================
# 1. Define the Problem
# ==========================================
def objective_function(x):
    """
    Rastrigin Function: Global minimum at x = [0, 0, ...] with f(x) = 0.
    Search domain typically: [-5.12, 5.12]
    """
    A = 10
    n = len(x)
    return A * n + np.sum(x**2 - A * np.cos(2 * np.pi * x))

def calculate_fitness(x):
    """
    Fitness function for minimization: higher fitness is better.
    """
    val = objective_function(x)
    return 1.0 / (1.0 + val)

# ==========================================
# 2. Initialize Parameters
# ==========================================
POP_SIZE = 50           # Number of individuals in the population
NUM_GENES = 5           # Dimension of the problem (number of variables)
GENE_BOUNDS = (-5.12, 5.12)  # Search space range
MUTATION_RATE = 0.1     # Probability of mutating each gene
CROSSOVER_RATE = 0.8    # Probability of crossover
GENERATIONS = 100       # Maximum iterations

# ==========================================
# 3. Initialize Population
# ==========================================
def create_individual(num_genes, bounds):
    # Genetic sequence encoded as floating-point vector (chromosomes)
    return np.random.uniform(bounds[0], bounds[1], size=num_genes)

def create_population(pop_size, num_genes, bounds):
    return [create_individual(num_genes, bounds) for _ in range(pop_size)]

# ==========================================
# 8. Gene Expression (Translation Layer)
# ==========================================
def express_genes(chromosome):
    """
    Translates raw genetic sequence into candidate solution space.
    Clamps genotype values strictly within domain boundaries.
    """
    return np.clip(chromosome, GENE_BOUNDS[0], GENE_BOUNDS[1])

# ==========================================
# 5. Selection
# ==========================================
def roulette_wheel_selection(population, fitnesses):
    """
    Selects one parent using Roulette Wheel Selection based on fitness.
    """
    total_fitness = np.sum(fitnesses)
    if total_fitness == 0:
        probs = np.ones(len(population)) / len(population)
    else:
        probs = fitnesses / total_fitness

    selected_idx = np.random.choice(len(population), p=probs)
    return population[selected_idx].copy()

# ==========================================
# 6. Crossover
# ==========================================
def crossover(parent1, parent2, crossover_rate):
    """
    Performs single-point crossover between two parents.
    """
    if np.random.rand() < crossover_rate:
        point = np.random.randint(1, len(parent1))
        child1 = np.concatenate([parent1[:point], parent2[point:]])
        child2 = np.concatenate([parent2[:point], parent1[point:]])
        return child1, child2
    return parent1.copy(), parent2.copy()

# ==========================================
# 7. Mutation
# ==========================================
def mutate(chromosome, mutation_rate, bounds):
    """
    Applies Gaussian mutation to introduce genetic variability.
    """
    mutated = chromosome.copy()
    for i in range(len(mutated)):
        if np.random.rand() < mutation_rate:
            # Gaussian noise centered at current gene value
            noise = np.random.normal(0, 1.0)
            mutated[i] += noise
    return mutated

# ==========================================
# Main Execution: 9. Iterate & 10. Output Best Solution
# ==========================================
def run_gene_expression_algorithm():
    population = create_population(POP_SIZE, NUM_GENES, GENE_BOUNDS)

    best_overall_solution = None
    best_overall_fitness = -1.0
    best_overall_cost = float('inf')

    print(f"Starting GEA Optimization (Dimensions: {NUM_GENES}, Population: {POP_SIZE}, Generations: {GENERATIONS})\n")

    for gen in range(1, GENERATIONS + 1):
        # Step 8 & Step 4: Gene Expression & Fitness Evaluation
        expressed_solutions = [express_genes(ind) for ind in population]
        fitnesses = np.array([calculate_fitness(sol) for sol in expressed_solutions])
        costs = np.array([objective_function(sol) for sol in expressed_solutions])

        # Step 10: Track Best Solution
        gen_best_idx = np.argmin(costs)
        if fitnesses[gen_best_idx] > best_overall_fitness:
            best_overall_fitness = fitnesses[gen_best_idx]
            best_overall_cost = costs[gen_best_idx]
            best_overall_solution = expressed_solutions[gen_best_idx].copy()

        if gen % 10 == 0 or gen == 1:
            print(f"Gen {gen:3d} | Best Cost: {best_overall_cost:.6f} | Best Fitness: {best_overall_fitness:.6f}")

        # Step 5: Selection
        new_population = []

        # Elitism: retain the best individual from the current generation
        new_population.append(best_overall_solution.copy())

        # Step 6 & 7: Crossover & Mutation to populate remaining slots
        while len(new_population) < POP_SIZE:
            parent1 = roulette_wheel_selection(population, fitnesses)
            parent2 = roulette_wheel_selection(population, fitnesses)

            child1, child2 = crossover(parent1, parent2, CROSSOVER_RATE)

            child1 = mutate(child1, MUTATION_RATE, GENE_BOUNDS)
            child2 = mutate(child2, MUTATION_RATE, GENE_BOUNDS)

            new_population.append(child1)
            if len(new_population) < POP_SIZE:
                new_population.append(child2)

        population = new_population

    print("\n" + "="*50)
    print("OPTIMIZATION COMPLETED")
    print("="*50)
    print(f"Best Solution Vector Found: {np.round(best_overall_solution, 5)}")
    print(f"Objective Function Value f(x): {best_overall_cost:.8f}")
    return best_overall_solution, best_overall_cost

if __name__ == "__main__":
    run_gene_expression_algorithm()

Starting GEA Optimization (Dimensions: 5, Population: 50, Generations: 100)

Gen   1 | Best Cost: 25.755400 | Best Fitness: 0.037376
Gen  10 | Best Cost: 10.657911 | Best Fitness: 0.085779
Gen  20 | Best Cost: 4.986224 | Best Fitness: 0.167050
Gen  30 | Best Cost: 2.930547 | Best Fitness: 0.254418
Gen  40 | Best Cost: 1.107667 | Best Fitness: 0.474458
Gen  50 | Best Cost: 0.447786 | Best Fitness: 0.690710
Gen  60 | Best Cost: 0.447786 | Best Fitness: 0.690710
Gen  70 | Best Cost: 0.324910 | Best Fitness: 0.754768
Gen  80 | Best Cost: 0.324910 | Best Fitness: 0.754768
Gen  90 | Best Cost: 0.294376 | Best Fitness: 0.772573
Gen 100 | Best Cost: 0.233130 | Best Fitness: 0.810944

OPTIMIZATION COMPLETED
Best Solution Vector Found: [-0.02796  0.00405 -0.00152  0.01243 -0.01492]
Objective Function Value f(x): 0.23313015
